# SteamRec – End-to-end notebook (Content-based + ML.NET collaborative)

Deze notebook is **volledig uitvoerbaar** (cel-per-cel) en behandelt:
- imports
- data-analyse
- selectie, exploratie en preprocessing (garbage in – garbage out)
- ML pipeline (train/test splits, cross-validation, evaluatie)
- hyperparameter-optimalisatie
- evaluatie (Precision/Recall, MAE, RMSE)
- forecasting-gebaseerde aanbevelingen (seizoensgebonden trends)
- visualisatie

Daarnaast tonen we hoe de **SteamRec.Core** en **SteamRec.ML** projecten gebruikt kunnen worden met dezelfde data.


## 0) Imports
Installeer indien nodig: `pip install pandas numpy pymongo scikit-learn scipy matplotlib seaborn`


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from pymongo import MongoClient

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split, KFold
from sklearn.metrics import precision_score, recall_score, mean_absolute_error, mean_squared_error
from sklearn.decomposition import TruncatedSVD
from scipy import sparse


## 1) Data laden uit MongoDB
De volgende cel gebruikt de opgegeven MongoDB snippet.


In [ ]:
MONGO_URI = os.getenv("MONGODB_CONNECTION_STRING", "mongodb://localhost:27017")
DB_NAME = "steamrec-test"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]

games = pd.DataFrame(list(db["games"].find({})))
interactions = pd.DataFrame(list(db["interactions"].find({})))

print(games.shape, interactions.shape)
games.head()


## 2) Schema-normalisatie helpers
We mappen kolomnamen naar een standaard schema zodat de rest van de notebook altijd werkt.


In [ ]:
def find_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_games(df):
    df = df.copy()
    col_app = find_col(df, ["app_id", "appId", "appid"])
    col_name = find_col(df, ["name", "title"])
    col_genres = find_col(df, ["genres", "genre"])
    col_tags = find_col(df, ["tags", "tag"])
    col_categories = find_col(df, ["categories", "category"])
    col_review_score = find_col(df, ["review_score_adj", "reviewScoreAdj", "review_score"])
    col_review_volume = find_col(df, ["review_volume_log", "reviewVolumeLog", "review_volume"])

    df.rename(columns={
        col_app: "app_id",
        col_name: "name",
        col_genres: "genres",
        col_tags: "tags",
        col_categories: "categories",
        col_review_score: "review_score_adj",
        col_review_volume: "review_volume_log",
    }, inplace=True)

    for col in ["genres", "tags", "categories"]:
        if col not in df.columns:
            df[col] = [[] for _ in range(len(df))]
        else:
            # Zorg dat dit altijd list-achtig is
            df[col] = df[col].apply(lambda x: x if isinstance(x, list) else ([] if pd.isna(x) else [str(x)]))

    for col in ["review_score_adj", "review_volume_log"]:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["app_id"])
    df["app_id"] = df["app_id"].astype(int)

    if "name" not in df.columns:
        df["name"] = df["app_id"].astype(str)

    return df

def normalize_interactions(df):
    df = df.copy()
    col_user = find_col(df, ["steam_id", "steamId", "user_id", "userId"])
    col_app = find_col(df, ["app_id", "appId", "appid"])
    col_play_forever = find_col(df, ["playtime_forever", "playtimeForever"])
    col_play_2w = find_col(df, ["playtime_2weeks", "playtime2weeks", "playtime2Weeks"])
    col_ts = find_col(df, ["timestamp", "created_at", "createdAt"])

    df.rename(columns={
        col_user: "steam_id",
        col_app: "app_id",
        col_play_forever: "playtime_forever",
        col_play_2w: "playtime_2weeks",
        col_ts: "timestamp",
    }, inplace=True)

    for col in ["playtime_forever", "playtime_2weeks"]:
        if col not in df.columns:
            df[col] = 0.0
        df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0.0)

    df["app_id"] = pd.to_numeric(df["app_id"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["app_id", "steam_id"])
    df["app_id"] = df["app_id"].astype(int)
    df["steam_id"] = df["steam_id"].astype(str)

    if "timestamp" in df.columns:
        df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

    return df


## 3) Data-analyse & exploratie
We bekijken de kwaliteit van de data en basale statistieken.


In [ ]:
games_norm = normalize_games(games)
interactions_norm = normalize_interactions(interactions)

print("Games", games_norm.shape)
print("Interactions", interactions_norm.shape)

display(games_norm.head())
display(interactions_norm.head())


In [ ]:
def missing_report(df):
    return (df.isna().sum().sort_values(ascending=False) / len(df)).to_frame('missing_ratio')

display(missing_report(games_norm).head(10))
display(missing_report(interactions_norm).head(10))


### Visualisaties – verdeling playtime


In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(interactions_norm["playtime_forever"].fillna(0), bins=50, log_scale=(False, True))
plt.title("Verdeling playtime_forever (log y)")
plt.xlabel("Playtime")
plt.ylabel("Aantal interacties")
plt.show()


## 4) Selectie, cleaning & preprocessing (Garbage in – Garbage out)
We filteren ongeldige records en maken het signaal robuuster.


In [ ]:
games_clean = games_norm.copy()
interactions_clean = interactions_norm.copy()

# Filter records zonder app_id of steam_id
interactions_clean = interactions_clean.dropna(subset=["steam_id", "app_id"])

# Zero-interacties verwijderen (geen signaal)
interactions_clean = interactions_clean[(interactions_clean["playtime_forever"] > 0) | (interactions_clean["playtime_2weeks"] > 0)]

# Impliciete rating zoals in SteamRec.ML (log-scale + recente bonus)
RECENT_BONUS = 0.35
interactions_clean["rating"] = (
    np.log1p(interactions_clean["playtime_forever"])
    + RECENT_BONUS * np.log1p(interactions_clean["playtime_2weeks"])
)

interactions_clean = interactions_clean[interactions_clean["rating"] > 0]

print(interactions_clean.shape)
interactions_clean.head()


## 5) Content-based features (SteamRec.Core concept)
We bouwen een TF-IDF representatie op basis van genres/tags/categories om content-similarity te berekenen.


In [ ]:
def combine_text_features(row):
    parts = []
    for col in ["genres", "tags", "categories"]:
        vals = row.get(col, [])
        if isinstance(vals, list):
            parts.extend([str(v) for v in vals])
        elif pd.notna(vals):
            parts.append(str(vals))
    return " \n".join(parts)

games_clean["text_features"] = games_clean.apply(combine_text_features, axis=1)

tfidf = TfidfVectorizer(min_df=1)
content_matrix = tfidf.fit_transform(games_clean["text_features"])

games_clean[["app_id", "name"]].head()


## 6) Train/test split per gebruiker (holdout)
We houden per gebruiker één interactie apart als test.


In [ ]:
def leave_one_out_split(df, user_col="steam_id", item_col="app_id"):
    test_rows = []
    train_rows = []
    for user_id, group in df.groupby(user_col):
        if len(group) < 2:
            train_rows.append(group)
            continue
        test_row = group.sample(1, random_state=42)
        train_row = group.drop(test_row.index)
        test_rows.append(test_row)
        train_rows.append(train_row)
    train_df = pd.concat(train_rows, ignore_index=True)
    test_df = pd.concat(test_rows, ignore_index=True) if test_rows else pd.DataFrame(columns=df.columns)
    return train_df, test_df

train_interactions, test_interactions = leave_one_out_split(interactions_clean)
print(train_interactions.shape, test_interactions.shape)


## 7) Content-based aanbevelingen (evaluatie)
We maken een gebruikersprofiel uit train-interacties en evalueren op de test-item(s).


In [ ]:
# Mapping app_id -> index in content_matrix
app_id_to_idx = {app_id: idx for idx, app_id in enumerate(games_clean["app_id"]) }
idx_to_app_id = {idx: app_id for app_id, idx in app_id_to_idx.items()}

def recommend_content_for_user(user_id, train_df, top_k=10):
    user_items = train_df[train_df["steam_id"] == user_id]["app_id"].unique().tolist()
    user_items = [i for i in user_items if i in app_id_to_idx]
    if not user_items:
        return []

    user_vec = content_matrix[[app_id_to_idx[i] for i in user_items]].mean(axis=0)
    sims = cosine_similarity(user_vec, content_matrix).ravel()

    # Exclude already seen items
    for i in user_items:
        sims[app_id_to_idx[i]] = -1

    top_idx = np.argsort(sims)[::-1][:top_k]
    return [idx_to_app_id[i] for i in top_idx]

def precision_recall_at_k(test_df, train_df, k=10):
    precisions = []
    recalls = []

    for user_id, group in test_df.groupby("steam_id"):
        true_items = set(group["app_id"])
        recs = set(recommend_content_for_user(user_id, train_df, top_k=k))
        if not recs:
            continue
        tp = len(true_items & recs)
        precisions.append(tp / k)
        recalls.append(tp / len(true_items))

    return float(np.mean(precisions)), float(np.mean(recalls))

prec_k, rec_k = precision_recall_at_k(test_interactions, train_interactions, k=10)
print("Content-based Precision@10", prec_k)
print("Content-based Recall@10", rec_k)


## 8) Collaborative filtering (analoge pipeline aan SteamRec.ML)
We bouwen een user-item matrix en gebruiken matrix-factorisatie via TruncatedSVD.


In [ ]:
# Encode users/items
user_ids = train_interactions["steam_id"].unique()
item_ids = train_interactions["app_id"].unique()

user_to_idx = {u: i for i, u in enumerate(user_ids)}
item_to_idx = {i: j for j, i in enumerate(item_ids)}

rows = train_interactions["steam_id"].map(user_to_idx)
cols = train_interactions["app_id"].map(item_to_idx)
data = train_interactions["rating"].values

user_item_matrix = sparse.csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(item_ids)))

# Train SVD model
svd = TruncatedSVD(n_components=50, random_state=42)
user_factors = svd.fit_transform(user_item_matrix)
item_factors = svd.components_.T

def recommend_cf_for_user(user_id, top_k=10):
    if user_id not in user_to_idx:
        return []
    u_idx = user_to_idx[user_id]
    scores = user_factors[u_idx] @ item_factors.T

    # Exclude already seen items
    seen = set(train_interactions[train_interactions["steam_id"] == user_id]["app_id"])
    for item in seen:
        if item in item_to_idx:
            scores[item_to_idx[item]] = -1

    top_idx = np.argsort(scores)[::-1][:top_k]
    return [item_ids[i] for i in top_idx]

def precision_recall_at_k_cf(test_df, train_df, k=10):
    precisions = []
    recalls = []
    for user_id, group in test_df.groupby("steam_id"):
        true_items = set(group["app_id"])
        recs = set(recommend_cf_for_user(user_id, top_k=k))
        if not recs:
            continue
        tp = len(true_items & recs)
        precisions.append(tp / k)
        recalls.append(tp / len(true_items))
    return float(np.mean(precisions)), float(np.mean(recalls))

prec_cf, rec_cf = precision_recall_at_k_cf(test_interactions, train_interactions, k=10)
print("Collaborative Precision@10", prec_cf)
print("Collaborative Recall@10", rec_cf)


### Evaluatie (MAE, RMSE)
We vergelijken voorspelde scores met werkelijke impliciete ratings.


In [ ]:
def predict_score(user_id, app_id):
    if user_id not in user_to_idx or app_id not in item_to_idx:
        return 0.0
    return float(user_factors[user_to_idx[user_id]] @ item_factors[item_to_idx[app_id]])

y_true = []
y_pred = []
for _, row in test_interactions.iterrows():
    y_true.append(row["rating"])
    y_pred.append(predict_score(row["steam_id"], row["app_id"]))

mae = mean_absolute_error(y_true, y_pred) if y_true else 0.0
rmse = mean_squared_error(y_true, y_pred, squared=False) if y_true else 0.0

print("MAE", mae)
print("RMSE", rmse)


## 9) Cross-validation (voorbeeld)
We gebruiken K-fold op gebruikers om de stabiliteit van het collaborative model te meten.


In [ ]:
def cv_evaluate(interactions_df, k_folds=3, n_components=30):
    users = interactions_df["steam_id"].unique()
    kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)
    scores = []

    for train_idx, test_idx in kf.split(users):
        train_users = users[train_idx]
        test_users = users[test_idx]

        train_df = interactions_df[interactions_df["steam_id"].isin(train_users)]
        test_df = interactions_df[interactions_df["steam_id"].isin(test_users)]

        user_ids = train_df["steam_id"].unique()
        item_ids = train_df["app_id"].unique()
        user_to_idx = {u: i for i, u in enumerate(user_ids)}
        item_to_idx = {i: j for j, i in enumerate(item_ids)}

        rows = train_df["steam_id"].map(user_to_idx)
        cols = train_df["app_id"].map(item_to_idx)
        data = train_df["rating"].values
        matrix = sparse.csr_matrix((data, (rows, cols)), shape=(len(user_ids), len(item_ids)))

        svd = TruncatedSVD(n_components=n_components, random_state=42)
        user_f = svd.fit_transform(matrix)
        item_f = svd.components_.T

        preds = []
        actuals = []
        for _, row in test_df.iterrows():
            if row["steam_id"] in user_to_idx and row["app_id"] in item_to_idx:
                score = float(user_f[user_to_idx[row["steam_id"]]] @ item_f[item_to_idx[row["app_id"]]])
                preds.append(score)
                actuals.append(row["rating"])

        if actuals:
            rmse = mean_squared_error(actuals, preds, squared=False)
            scores.append(rmse)

    return scores

cv_scores = cv_evaluate(interactions_clean, k_folds=3, n_components=30)
print("CV RMSE scores", cv_scores)
print("CV RMSE mean", np.mean(cv_scores) if cv_scores else None)


## 10) Hyperparameter-optimalisatie
We vergelijken verschillende aantallen SVD-componenten en TF-IDF instellingen.


In [ ]:
svd_candidates = [20, 40, 60]
tfidf_min_df = [1, 2]

results = []
for n_comp in svd_candidates:
    cv_scores = cv_evaluate(interactions_clean, k_folds=3, n_components=n_comp)
    results.append({"n_components": n_comp, "rmse_mean": np.mean(cv_scores) if cv_scores else np.nan})

results_df = pd.DataFrame(results)
display(results_df.sort_values("rmse_mean"))


## 11) Forecasting-gebaseerde aanbevelingen (seizoensgebonden trends)
We analyseren maandelijkse trends in playtime. Indien timestamps ontbreken, wordt deze stap overgeslagen.


In [ ]:
if "timestamp" in interactions_clean.columns and interactions_clean["timestamp"].notna().any():
    monthly = (interactions_clean
               .dropna(subset=["timestamp"])
               .assign(month=lambda df: df["timestamp"].dt.to_period("M").dt.to_timestamp())
               .groupby("month")["playtime_forever"]
               .sum()
               .reset_index())

    monthly["rolling_mean"] = monthly["playtime_forever"].rolling(window=3, min_periods=1).mean()

    plt.figure(figsize=(9, 4))
    sns.lineplot(data=monthly, x="month", y="playtime_forever", label="Playtime")
    sns.lineplot(data=monthly, x="month", y="rolling_mean", label="3M rolling mean")
    plt.title("Maandelijkse playtime trends")
    plt.show()
else:
    print("Geen timestamps beschikbaar in interactions; forecasting stap overgeslagen.")


## 12) Visualisatie van top-N aanbevelingen
We vergelijken content-based vs collaborative top-N aanbevelingen voor een voorbeeldgebruiker.


In [ ]:
example_user = train_interactions["steam_id"].iloc[0] if len(train_interactions) else None

if example_user:
    topn_content = recommend_content_for_user(example_user, train_interactions, top_k=10)
    topn_cf = recommend_cf_for_user(example_user, top_k=10)

    topn_content_df = pd.DataFrame({"app_id": topn_content, "model": "content"})
    topn_cf_df = pd.DataFrame({"app_id": topn_cf, "model": "collaborative"})

    topn_df = pd.concat([topn_content_df, topn_cf_df])

    plt.figure(figsize=(8, 4))
    sns.countplot(data=topn_df, x="app_id", hue="model")
    plt.title(f"Top-N aanbevelingen voor user {example_user}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print("Geen gebruikersdata beschikbaar om top-N te visualiseren.")


## 13) SteamRec.Core & SteamRec.ML – integratie
De volgende sectie toont hoe dezelfde data gebruikt kan worden in C#. Dit is een direct toepasbare stap in jullie projecten.

### Export naar CSV voor C# projecten
We exporteren de opgeschoonde data zodat C# deze eenvoudig kan inladen.


In [ ]:
export_dir = Path('data_exports')
export_dir.mkdir(exist_ok=True)

games_clean.to_csv(export_dir / 'games_clean.csv', index=False)
interactions_clean.to_csv(export_dir / 'interactions_clean.csv', index=False)

print("Exports written to", export_dir.resolve())


### Content-based (SteamRec.Core) – voorbeeld C#
```csharp
using SteamRec.Core;

// Laad games_clean.csv in en map naar GameRecord (vb. met CsvHelper).
var games = LoadGameRecords("data_exports/games_clean.csv");

var recommender = new ContentBasedRecommender(games);
var similar = recommender.RecommendSimilar(appId: 570, topN: 10);
var personal = recommender.RecommendForLiked(new[] { 570, 730 }, topN: 20);
```

### Collaborative filtering (SteamRec.ML) – voorbeeld C#
```csharp
using SteamRec.ML;

var rows = LoadInteractions("data_exports/interactions_clean.csv");
var cf = new CollaborativeFilteringRecommender();

cf.TrainFromRows(rows);

var recommendations = cf.RecommendForUser("7656119...", candidateAppIds, excludeAppIds, topN: 20);
```

### Tip – build je C# projecten
```bash
dotnet build csharp/SteamRec.Core
dotnet build csharp/SteamRec.ML
```
